### Planning & Reasoning

**ReAct** thinks one step at a time — reactive. It reacts to each observation before deciding what to do next. This makes it flexible but slower and prone to loops.

**Planning** writes the full plan upfront, then executes it — proactive. It commits to a sequence of steps before touching any tool. This makes it faster, more predictable, and easier to debug.

In [1]:
import os
import logging
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from typing import List

In [2]:
os.environ['ANONYMIZED_TELEMETRY'] = 'False' 
logging.getLogger('httpx').setLevel(logging.WARNING)

In [3]:
load_dotenv()

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

Three Simple Tools

In [4]:
@tool
def search_crop_disease(query: str) -> str:
    '''
    Search the crop disease knowledge base for information and treatment.
    '''
    
    knowledge = {
        'cassava mosaic': 'Cassava Mosaic Disease: viral, causes yellowing and mottling of leaves. Control: use disease-free cuttings, plant resistant varieties, remove infected plants.',
        'maize smut':     'Maize Smut: fungal, causes grey or black galls on ears and stalks. Control: plant resistant hybrids, rotate crops, remove galls before they burst.',
        'rice blast':     'Rice Blast: fungal, causes diamond-shaped lesions on leaves. Control: use resistant varieties, balance nitrogen, apply fungicides when needed.',
    }
    q = query.lower()
    for key, value in knowledge.items():
        if key in q:
            return value
    return f'No knowledge found for: {query}'

@tool
def get_weather(city: str) -> str:
    '''
    Return the current weather for a city.
    '''
    
    fake = {
        'lagos': 'Sunny, 32°C, humidity 78%',
        'abuja': 'Cloudy, 28°C, humidity 65%',
        'kano':  'Hot and dry, 36°C, humidity 30%',
    }
    
    return fake.get(city.lower(), f'No weather data for {city}')

@tool
def get_farmer_record(farmer_id: str) -> str:
    """Look up a farmer's profile and registered crops."""
    farmers = {
        'F001': 'Adebayo Okafor — crops: cassava, maize — location: Oyo',
        'F002': 'Fatima Ibrahim — crops: rice, tomato — location: Kano',
        'F003': 'Chinedu Eze — crops: cassava, yam — location: Enugu',
    }
    return farmers.get(farmer_id.upper(), f'No farmer found with ID: {farmer_id}')


print('Three tools ready')
print(f'   - {search_crop_disease.name}')
print(f'   - {get_weather.name}')
print(f'   - {get_farmer_record.name}')

Three tools ready
   - search_crop_disease
   - get_weather
   - get_farmer_record


**The Plan Parser:** convert the plan string into structured data.

In [5]:
# Build a lookup: tool name -> tool object

TOOL_MAP = {
    t.name: t for t in [search_crop_disease, get_weather, get_farmer_record]
}

print('Tool map built:')
for name in TOOL_MAP:
    print(f'  - {name}')

Tool map built:
  - search_crop_disease
  - get_weather
  - get_farmer_record


Define the Pydantic Schema for a Plan

In [6]:
class PlanStep(BaseModel):
    tool: str = Field(description='Tool name: search_crop_disease, get_weather, or get_farmer_record')
    argument: str = Field(description='The input to pass to the tool')
    
class Plan(BaseModel):
    steps: List[PlanStep] = Field(description='Ordered list of steps to execute')
    
print('Pydantic schema defined')
print(f'PlanStep fields: {list(PlanStep.model_fields.keys())}')
print(f'Plan fields: {list(Plan.model_fields.keys())}')

Pydantic schema defined
PlanStep fields: ['tool', 'argument']
Plan fields: ['steps']


The Planner Prompt

In [7]:
planner_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a planning assistant. Given a user question, decide which tools to call and in what order.\n'
     '\n'
     'Available tools:\n'
     '- search_crop_disease: search crop disease info (input: disease name)\n'
     '- get_weather: current weather for a city (input: city name)\n'
     '- get_farmer_record: farmer profile and crops (input: farmer ID like F001)\n'
     '\n'
     'Rules:\n'
     '1. Return only the tools that are actually needed.\n'
     '2. Order matters — if a later step depends on an earlier step, order them correctly.\n'
     '3. If an argument depends on a previous step\'s result, use a placeholder like:\n'
     '     <CITY_FROM_STEP_1>, <CROP_FROM_STEP_1>, <FARMER_FROM_STEP_1>\n'
     '4. Only use the three tools listed above. Do not invent tools.\n'
     '5. If no tool is needed, return an empty steps list.'),
    ('human', 'Question: {question}'),
])

planner_chain = planner_prompt | llm.with_structured_output(Plan, method='json_schema')

print('Planner prompt updated with placeholder convention')

Planner prompt updated with placeholder convention


In [8]:
planner_chain = planner_prompt | llm.with_structured_output(Plan, method='json_schema')

print('Structured planner chain ready')


Structured planner chain ready


Test — Call the Planner

In [9]:
plan = planner_chain.invoke({'question': 'What is the weather in Lagos?'})

print('Type:', type(plan).__name__)
print('steps')

for i, step in enumerate(plan.steps):
    print(f'   [{i+1}] tool={step.tool!r}, argument={step.argument!r}')

Type: Plan
steps
   [1] tool='get_weather', argument='Lagos'


Test a Compound Question

In [10]:
plan = planner_chain.invoke({
    'question': 'Look up farmer F002 and check the weather in their city.'
})

print('Steps:')
for i, step in enumerate(plan.steps):
    print(f'  [{i+1}] {step.tool}({step.argument})')

Steps:
  [1] get_farmer_record(F002)
  [2] get_weather(<CITY_FROM_STEP_1>)


The Executor

In [11]:
import re

# Tool lookup: name → tool object
TOOL_MAP = {t.name: t for t in [search_crop_disease, get_weather, get_farmer_record]}

# Map which argument key each tool expects
ARG_KEY = {
    'search_crop_disease': 'query',
    'get_weather':         'city',
    'get_farmer_record':   'farmer_id',
}

def resolve_argument(argument: str, results: list) -> str:
    """Replace placeholders like <CITY_FROM_STEP_1> with actual values."""
    def substitute(match):
        placeholder = match.group(1)          # e.g. CITY_FROM_STEP_1
        parts = placeholder.split('_FROM_STEP_')
        if len(parts) != 2:
            return match.group(0)             # unrecognized → leave as-is
        step_num = int(parts[1]) - 1          # 1-indexed → 0-indexed
        if step_num >= len(results):
            return match.group(0)
        # Extract the value from the previous step's result
        previous = results[step_num]
        if parts[0] == 'CITY':
            # Look for city names in the result
            for city in ['Lagos', 'Abuja', 'Kano']:
                if city.lower() in previous.lower():
                    return city
        if parts[0] == 'CROP':
            for crop in ['cassava', 'maize', 'rice', 'tomato']:
                if crop in previous.lower():
                    return crop
        if parts[0] == 'FARMER':
            return previous[:60]              # fallback: return first 60 chars
        return match.group(0)

    return re.sub(r'<([A-Z_]+)>', substitute, argument)


def execute_plan(plan, verbose=True):
    """Run every step in the plan, substituting placeholders as needed."""
    results = []
    for i, step in enumerate(plan.steps):
        if verbose:
            print(f'\n── Step {i+1} ──')

        # Resolve placeholders using prior results
        argument = resolve_argument(step.argument, results)

        if verbose:
            print(f'   Tool:       {step.tool}')
            print(f'   Raw arg:    {step.argument}')
            if argument != step.argument:
                print(f'   Resolved:   {argument}')

        # Look up the tool
        tool = TOOL_MAP.get(step.tool)
        if tool is None:
            results.append(f'ERROR: unknown tool {step.tool}')
            continue

        # Run the tool
        arg_key = ARG_KEY[step.tool]
        result = tool.invoke({arg_key: argument})
        results.append(result)

        if verbose:
            print(f'   Result:     {result[:100]}')

    return results


print('Executor ready')

Executor ready


The Full Planning Agent

In [12]:
# def planning_agent(question: str) -> str:
#     """Plan, execute, and answer."""

#     # Phase 1: Plan
#     print('── PLANNING ──')
#     plan = planner_chain.invoke({'question': question})
#     for i, step in enumerate(plan.steps):
#         print(f'   [{i+1}] {step.tool}({step.argument})')

#     if not plan.steps:
#         print('   (no tools needed)')
#         return 'No tool needed.'

#     # Phase 2: Execute
#     print('\n── EXECUTING ──')
#     results = execute_plan(plan)

#     # Phase 3: Answer
#     print('\n── ANSWERING ──')
#     return results


# print('Planning agent ready')

Answer Synthesis

In [13]:
answer_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a helpful farming assistant. Answer the user\'s question '
     'using only the information from the tool results below. '
     'If the results do not contain the answer, say you don\'t know.'),
    ('human',
     'Question: {question}\n\n'
     'Tool results:\n{results}\n\n'
     'Answer:'),
])

answer_chain = answer_prompt | llm | StrOutputParser()

print('Answer synthesis chain ready')

Answer synthesis chain ready


Update planning_agent() to Use Answer Synthesis

In [14]:
def planning_agent(question: str) -> str:
    """Plan, execute, and answer with a natural-language response."""

    # Phase 1: Plan
    print('── PLANNING ──')
    plan = planner_chain.invoke({'question': question})
    for i, step in enumerate(plan.steps):
        print(f'   [{i+1}] {step.tool}({step.argument})')

    if not plan.steps:
        print('   (no tools needed)')
        return 'I can answer that without tools.'

    # Phase 2: Execute
    print('\n── EXECUTING ──')
    results = execute_plan(plan)

    # Phase 3: Synthesize answer with LLM
    print('\n── ANSWERING ──')
    results_text = '\n'.join(f'- {r}' for r in results)
    answer = answer_chain.invoke({
        'question': question,
        'results': results_text,
    })
    return answer


print('Planning agent upgraded with answer synthesis')

Planning agent upgraded with answer synthesis


Compound Question

In [15]:
answer = planning_agent('Look up farmer F002 and check the weather in their city.')
print('\n🤖 Answer:', answer)

── PLANNING ──


   [1] get_farmer_record(F002)
   [2] get_weather(<CITY_FROM_STEP_1>)

── EXECUTING ──

── Step 1 ──
   Tool:       get_farmer_record
   Raw arg:    F002
   Result:     Fatima Ibrahim — crops: rice, tomato — location: Kano

── Step 2 ──
   Tool:       get_weather
   Raw arg:    <CITY_FROM_STEP_1>
   Result:     No weather data for <CITY_FROM_STEP_1>

── ANSWERING ──

🤖 Answer: I don't know.
